# Inline train preprocess: churn from DAC MVP

Эта тетрадка повторяет логику `src/cvm_model/train/preprocess.py`, но разложена по ячейкам, чтобы её можно было запускать и отлаживать в Jupyter пошагово.

Основная идея:

1. Берём DAC-аудиторию в базовом месяце.
2. Размечаем таргет ухода из DAC в следующем месяце.
3. Собираем полный набор фичей через `utils.load_features`.
4. Проверяем missing features, выбросы, target distribution.
5. Опционально сохраняем датасет в S3 `input`.

## 0. Path and env

Если проект не установлен в kernel, `sys.path` позволит импортировать `cvm_model` из локальной папки `src`.

Если есть `.env`, раскомментируй `%dotenv`.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/underplums/Documents/work/organic-return-dac')
src_path = project_root / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root, src_path

In [ ]:
# %load_ext dotenv
# %dotenv /Users/underplums/Documents/work/organic-return-dac/.env

## 1. Imports

In [ ]:
from datetime import datetime
from pathlib import Path

import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import magpie.sql_utils as su

from cvm_model.io import State
import cvm_model.utils as utils
import cvm_model.sql_my as sql
from cvm_model.parameters import (
    features_for_outliers,
    aud_table,
    fav_omni_features_table,
    preperiod_months,
    uplift_rate_prefix,
    aud_suffix,
    features,
    input_suffix,
    template,
)

## 2. Runtime flags

`WRITE_TO_S3 = False` позволяет сначала собрать датасет локально в памяти и не трогать S3. Когда всё проверено, можно поставить `True`.

In [ ]:
event_timestamp = datetime(2025, 11, 1)

WRITE_TO_S3 = False
CLEAN_S3_INPUT = False
USE_TEMP_AUD_TABLE = True
CLEANUP_TEMP_OBJECTS = False

# Для быстрой отладки можно включить sample после загрузки аудитории.
# Для полного прогона поставь None.
SAMPLE_N = 10_000

base_month = event_timestamp.date().replace(day=1).isoformat()
target_month = (pd.Timestamp(base_month) + pd.DateOffset(months=1)).date().isoformat()
feature_date = target_month

base_month, target_month, feature_date

## 3. State and connections

Если падает здесь, значит kernel не видит нужные env-переменные или инфраструктурные доступы.

In [ ]:
state = State.from_env()
engine = state.credentials.loyalty_gp.sa_engine
session = state.spark.session
s3 = su.get_s3_client()

engine, session

## 4. Resolve S3 input path

In [ ]:
input_prefix = state.settings.get_prefix(temp=True, suffix=input_suffix)
input_bucket = input_prefix.split('//')[1].split('/')[0]
input_key = '/'.join(input_prefix.split('//')[1].split('/')[1:])
input_path = template.format(bucket=input_bucket, prefix=input_key)

input_bucket, input_key, input_path

In [ ]:
if CLEAN_S3_INPUT:
    su.remove_from_s3(input_key, dry_run=False)
    files = su.list_all_s3_objects(s3, input_key)
    assert len(files) == 0, 'Не очистилась папка для датасета!'
else:
    print('Skip S3 input cleanup')

## 5. Audience

Аудитория: клиенты, которые являются DAC в `base_month` по новой логике.

In [ ]:
aud_query_raw = sql.aud_query.format(base_month=base_month)

df = utils.get_df(engine, aud_query_raw).fillna(0)
df = df.astype({col: np.int32 for col in {'contact_id'} & set(df.columns)})

assert len(df) > 0
assert 'contact_id' in df.columns
assert df['contact_id'].nunique() == len(df)

print(df.shape)
df.head()

In [ ]:
if SAMPLE_N is not None and len(df) > SAMPLE_N:
    df = df.sample(SAMPLE_N, random_state=42).reset_index(drop=True)
    print('Sampled audience:', df.shape)

df.head()

## 6. Upload audience to temp GP table

Так же делает исходный preprocess: временная таблица ускоряет последующие SQL-запросы.

In [ ]:
if USE_TEMP_AUD_TABLE:
    utils.upload_df(engine, pd.DataFrame(df['contact_id']).astype(int), aud_table)
    aud_query = f'select * from {aud_table}'
else:
    aud_query = aud_query_raw

aud_query[:500]

## 7. Recency

Recency оставляем как фичу. Не фильтруем по `login_recency`, потому что новая DAC-логика включает PWA и offline virtual card.

In [ ]:
query_kwargs = dict(
    aud=aud_query,
    date=feature_date,
    month=preperiod_months[0],
)

df_part = utils.get_df(engine, sql.recency_query.format(**query_kwargs))
assert len(df_part) / len(df) >= 0.99, 'Не выгрузилась recency-таблица более чем для 1% аудитории.'

df = df.merge(df_part, on='contact_id', how='left')

print(df.shape)
df[['contact_id', 'cheque_recency', 'login_recency', 'omni_qr_recency', 'omni_features_recency']].head()

## 8. Refresh temp audience table

Повторяем исходный паттерн: после возможных фильтров обновляем таблицу аудитории.

In [ ]:
if USE_TEMP_AUD_TABLE:
    utils.upload_df(engine, pd.DataFrame(df['contact_id']).astype(int), aud_table)
    aud_query = f'select * from {aud_table}'

aud_query[:500]

## 9. Target

`target_churn_from_dac = 1`: клиент был DAC в базовом месяце и не DAC в следующем.

In [ ]:
query_kwargs = dict(
    aud=aud_query,
    target_month=target_month,
)

df_part = utils.get_df(engine, sql.target_query.format(**query_kwargs)).fillna(0)
df_part = df_part.astype({col: np.int32 for col in {'target_trns', 'target_login'} & set(df_part.columns)})
df_part['target_dac'] = df_part['target_trns'] * df_part['target_login']

df = df.merge(df_part, on='contact_id')

print(df.shape)
display(df['target_churn_from_dac'].value_counts(dropna=False).to_frame('cnt'))
display(df['target_churn_from_dac'].value_counts(normalize=True, dropna=False).to_frame('share'))
df.head()

## 10. Full feature loading

Это тот же `utils.load_features`, что использует исходный preprocess. Здесь будут тяжёлые запросы.

In [ ]:
aud_prefix = state.settings.get_prefix(temp=True, suffix=aud_suffix)

df = utils.load_features(
    engine=engine,
    session=session,
    df=df,
    aud_query=aud_query,
    date=feature_date,
    fav_omni_features_table=fav_omni_features_table,
    preperiod_months=preperiod_months,
    aud_prefix=aud_prefix,
    uplift_rate_prefix=uplift_rate_prefix,
)

print(df.shape)
df.head()

## 11. Feature checks

In [ ]:
missing_features = sorted(set(features) - set(df.columns))
missing_features

In [ ]:
assert len(missing_features) == 0, f'В датасете не хватает фичей: {missing_features}'

display(df[features + ['target_churn_from_dac']].isna().mean().sort_values(ascending=False).to_frame('null_share'))
display(df[features].describe().T)

## 12. EDA and Feature Diagnostics

Блок для анализа качества датасета и первичного отбора фичей.

Что смотрим:

- распределение таргета;
- долю пропусков и нулей;
- PhiK-корреляции, включая связь признаков с таргетом;
- Pearson/Spearman-корреляции для числовых фичей;
- распределения топ-фичей;
- target rate по децилям топ-фичей.

In [ ]:
import sys
import subprocess

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import phik
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'phik'])
    import phik

sns.set_theme(style='whitegrid')

target_col = 'target_churn_from_dac'
assert target_col in df.columns, f'В df нет таргета {target_col}'

numeric_features = [f for f in features if f in df.columns and pd.api.types.is_numeric_dtype(df[f])]
eda_df = df[numeric_features + [target_col]].copy()

print('df shape:', df.shape)
print('numeric features:', len(numeric_features))
print('target:', target_col)

### 12.1 Target Distribution

In [ ]:
target_counts = df[target_col].value_counts(dropna=False).sort_index()
target_share = df[target_col].value_counts(normalize=True, dropna=False).sort_index()
target_report = pd.concat([target_counts.rename('cnt'), target_share.rename('share')], axis=1)
display(target_report)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x=target_col, ax=axes[0], color='#4C78A8')
axes[0].set_title('Target count')
axes[0].set_xlabel(target_col)
axes[0].set_ylabel('count')

axes[1].pie(
    target_counts.values,
    labels=[str(x) for x in target_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    colors=['#72B7B2', '#E45756'][:len(target_counts)],
)
axes[1].set_title('Target share')
plt.tight_layout()
plt.show()

### 12.2 Missing, Zero And Basic Stats

In [ ]:
eda_report = pd.DataFrame(index=numeric_features)
eda_report['dtype'] = df[numeric_features].dtypes.astype(str)
eda_report['missing_share'] = df[numeric_features].isna().mean()
eda_report['zero_share'] = (df[numeric_features] == 0).mean()
eda_report['nunique'] = df[numeric_features].nunique(dropna=True)
eda_report['mean'] = df[numeric_features].mean(numeric_only=True)
eda_report['std'] = df[numeric_features].std(numeric_only=True)
eda_report['p01'] = df[numeric_features].quantile(0.01, numeric_only=True)
eda_report['p50'] = df[numeric_features].quantile(0.50, numeric_only=True)
eda_report['p99'] = df[numeric_features].quantile(0.99, numeric_only=True)

display(eda_report.sort_values(['missing_share', 'zero_share'], ascending=False))

bad_features = eda_report[(eda_report['missing_share'] > 0.95) | (eda_report['nunique'] <= 1)]
display(bad_features.sort_values(['missing_share', 'nunique'], ascending=[False, True]))

In [ ]:
plot_report = eda_report.sort_values('missing_share', ascending=False).head(30).reset_index(names='feature')

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.barplot(data=plot_report, y='feature', x='missing_share', ax=axes[0], color='#E45756')
axes[0].set_title('Top missing share')
axes[0].set_xlabel('missing share')
axes[0].set_ylabel('')

plot_zero = eda_report.sort_values('zero_share', ascending=False).head(30).reset_index(names='feature')
sns.barplot(data=plot_zero, y='feature', x='zero_share', ax=axes[1], color='#72B7B2')
axes[1].set_title('Top zero share')
axes[1].set_xlabel('zero share')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### 12.3 PhiK Correlation Matrix

In [ ]:
# PhiK может быть тяжёлым, поэтому для матрицы используем sample.
phik_sample_n = min(len(eda_df), 10_000)
phik_df = eda_df.sample(phik_sample_n, random_state=42) if len(eda_df) > phik_sample_n else eda_df.copy()

# Убираем совсем пустые/константные признаки, PhiK их не любит.
phik_cols = [c for c in phik_df.columns if phik_df[c].notna().sum() > 0 and phik_df[c].nunique(dropna=True) > 1]
phik_df = phik_df[phik_cols]
interval_cols = [c for c in phik_cols if c != target_col]

phik_matrix = phik_df.phik_matrix(interval_cols=interval_cols)

phik_target = (
    phik_matrix[target_col]
    .drop(target_col, errors='ignore')
    .sort_values(ascending=False)
)

display(phik_target.to_frame('phik_with_target').head(30))

In [ ]:
top_phik_features = phik_target.head(30).index.tolist()
heatmap_cols = top_phik_features + [target_col]

plt.figure(figsize=(16, 14))
sns.heatmap(
    phik_matrix.loc[heatmap_cols, heatmap_cols],
    cmap='viridis',
    vmin=0,
    vmax=1,
    square=False,
    cbar_kws={'label': 'PhiK'},
)
plt.title('PhiK correlation matrix: top features by target relation')
plt.tight_layout()
plt.show()

### 12.4 Pearson And Spearman Correlations

In [ ]:
pearson_target = (
    eda_df.corr(method='pearson', numeric_only=True)[target_col]
    .drop(target_col)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
spearman_target = (
    eda_df.corr(method='spearman', numeric_only=True)[target_col]
    .drop(target_col)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

corr_target_report = pd.concat(
    [pearson_target.rename('pearson'), spearman_target.rename('spearman'), phik_target.rename('phik')],
    axis=1,
).sort_values('phik', ascending=False)

display(corr_target_report.head(40))

In [ ]:
top_corr_features = corr_target_report.head(25).index.tolist()
corr_for_heatmap = eda_df[top_corr_features].corr(method='spearman', numeric_only=True).abs()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_for_heatmap, cmap='mako', vmin=0, vmax=1)
plt.title('Spearman abs correlation: top target-related features')
plt.tight_layout()
plt.show()

upper = corr_for_heatmap.where(np.triu(np.ones(corr_for_heatmap.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'spearman_abs_corr'})
    .sort_values('spearman_abs_corr', ascending=False)
)
display(high_corr_pairs[high_corr_pairs['spearman_abs_corr'] >= 0.90].head(50))

### 12.5 Feature Distributions And Target Rate By Deciles

In [ ]:
plot_features = [f for f in phik_target.head(8).index if f in df.columns]

ncols = 2
nrows = int(np.ceil(len(plot_features) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, f in zip(axes, plot_features):
    sns.histplot(data=df, x=f, hue=target_col, bins=40, stat='density', common_norm=False, ax=ax)
    ax.set_title(f'Distribution: {f}')

for ax in axes[len(plot_features):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def feature_decile_report(data, feature, target=target_col, q=10):
    tmp = data[[feature, target]].copy()
    tmp = tmp[tmp[feature].notna()]
    if tmp[feature].nunique() < 2:
        return None
    tmp['bin'] = pd.qcut(tmp[feature], q=q, duplicates='drop')
    return (
        tmp.groupby('bin', observed=True)
        .agg(
            cnt=(target, 'size'),
            target_rate=(target, 'mean'),
            feature_min=(feature, 'min'),
            feature_max=(feature, 'max'),
        )
        .reset_index()
    )

decile_features = [f for f in phik_target.head(8).index if f in df.columns]
decile_reports = {}

for f in decile_features:
    rep = feature_decile_report(df, f)
    if rep is None:
        continue
    rep['feature'] = f
    rep['decile'] = range(1, len(rep) + 1)
    decile_reports[f] = rep

    fig, ax1 = plt.subplots(figsize=(10, 4))
    sns.lineplot(data=rep, x='decile', y='target_rate', marker='o', ax=ax1, color='#E45756')
    ax1.set_title(f'Target rate by decile: {f}')
    ax1.set_ylabel('target rate')
    ax1.set_xlabel('feature decile')
    ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

if decile_reports:
    display(pd.concat(decile_reports.values(), ignore_index=True))

### 12.6 Save EDA Tables Locally

In [ ]:
eda_dir = project_root / 'eda_churn_from_dac'
eda_dir.mkdir(parents=True, exist_ok=True)

eda_report.to_csv(eda_dir / 'feature_missing_zero_stats.csv')
corr_target_report.to_csv(eda_dir / 'feature_target_correlations.csv')
phik_matrix.to_csv(eda_dir / 'phik_matrix.csv')
high_corr_pairs.to_csv(eda_dir / 'high_spearman_corr_pairs.csv', index=False)

if decile_reports:
    pd.concat(decile_reports.values(), ignore_index=True).to_csv(eda_dir / 'feature_decile_reports.csv', index=False)

eda_dir

## 13. Outliers

Исходный preprocess удалял выбросы внутри churn-сегментов. У новой DAC-аудитории этих сегментов нет, поэтому считаем 0.999-квантиль по всей аудитории.

In [ ]:
q = 0.999
df['outlier'] = 0

for f in features_for_outliers:
    if f not in df.columns or df[f].notna().sum() == 0:
        continue
    t = np.nanquantile(df[f], q)
    df.loc[df[f] > t, 'outlier'] = 1

outlier_share = len(df[df['outlier'] == 1]) / len(df)
outlier_share

In [ ]:
assert outlier_share <= 0.01, 'Удалено как выбросы более 1% аудитории.'

df = df[df['outlier'] == 0].reset_index(drop=True)
df = df.drop(columns=['outlier'])

df.shape

## 14. Save local debug parquet

In [ ]:
local_path = project_root / 'df_churn_from_dac_debug.parquet'
df.to_parquet(local_path)
local_path

## 15. Optional S3 save

Включай только когда датасет проверен.

In [ ]:
if WRITE_TO_S3:
    print(f'Saving dataset to {input_bucket}/{input_key}')
    su.save_to_s3(df, input_key, input_type='df', bucket=input_bucket)
else:
    print('Skip S3 save')

## 16. Optional cleanup

In [ ]:
if CLEANUP_TEMP_OBJECTS:
    su.remove_from_s3(aud_prefix, dry_run=False)
    for table in [aud_table, fav_omni_features_table]:
        utils.execute_query(engine, f'drop table if exists {table}')
else:
    print('Skip cleanup')